In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [12]:
commodity   = "Fruits"
region      = "Southeast"
supply_stage = "Transit"
LAGS        = 24 # AutoReg lags (months)
 
# will try SARIMA model after AR model
# s=12 for monthly seasonality to account for climate cycles


### Data
Target columns: 'loss_percentage' in [0, 1]

   Planned datasets:
     - NOAA GHCN-Daily temperature anomalies
     - USDA ERS commodity loss estimates
     - FMCSA transit logs


In [16]:
# one of our datasets on loss percentage for commodity over different years

food_waste = pd.read_csv('Data.csv')

In [15]:
food_waste.head(1)

,m49_code,country,region,cpc_code,commodity,year,loss_percentage,loss_percentage_original,loss_quantity,activity,food_supply_stage,treatment,cause_of_loss,sample_size,method_data_collection,reference,url,notes
0,840,United States of America,NaN,1211.0,Asparagus,2019,9.0,9,NaN,NaN,Whole supply chain,NaN,NaN,NaN,National Accounts,"USDA, (2021), USDA Food Loss",https://www.ers.usda.gov/data-products/food-av...,NaN


In [ ]:
## check ACF/PACF
 
series = spoilage_df["spoilage_risk"]
 
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(series,  lags=36, ax=axes[0],
         title=f"ACF — {COMMODITY} Spoilage Risk")
plot_pacf(series, lags=36, ax=axes[1],
          title=f"PACF — {COMMODITY} Spoilage Risk")
plt.tight_layout()
plt.show()

In [ ]:
train_size = int(0.9 * len(spoilage_df))
train_data = spoilage_df.iloc[:train_size]
test_data  = spoilage_df.iloc[train_size:]
 
y_train = np.array(train_data["spoilage_risk"]).reshape(-1, 1)
y_test  = np.array(test_data["spoilage_risk"]).reshape(-1, 1)

## Inital model: try AR(24) to predict risk from past 24 months.

In [ ]:
ar_model   = AutoReg(y_train.flatten(), lags=LAGS, trend="c")
ar_results = ar_model.fit()
print(ar_results.summary())
 
ar_preds = ar_results.predict(
    start=len(y_train),
    end=len(y_train) + len(y_test) - 1,
    dynamic=False
)
 
ar_mae  = mean_absolute_error(y_test, ar_preds)
ar_rmse = np.sqrt(mean_squared_error(y_test, ar_preds))
print(f"MAE: {ar_mae:.4f}")
print(f"RMSE: {ar_rmse:.4f}")